# Native CLM v0 M3L-2 — Online Historical Address-State Integration

Canonical two-GPU formal workflow. GPU0 runs the matched M3R lineage-cosine control; GPU1 runs the online rank-32 address-state treatment. Formal seeds 74211/74212/74213 remain untouched until the explicit formal runner cell.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

BRANCH = 'codex/native-clm-v0-m3l2-online-address-state'
REPO = Path('/kaggle/working/mini-cells')
CHECKPOINT_DIR = Path('/kaggle/working/native-clm-v0-m1')
CHECKPOINT = CHECKPOINT_DIR / 'final-model.pt'
DATA = Path('/kaggle/working/native-clm-m3l2-data')
OUT = REPO / 'artifacts/experiments/native-clm-v0-m3l2-online-address-state'

def run(cmd, check=True, env=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=check, env=env)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/ArcheLabs/mini-cells.git', REPO])
else:
    os.chdir(REPO)
    run(['git', 'fetch', '--no-tags', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'])
    run(['git', 'checkout', BRANCH])
    run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'])
os.chdir(REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'])
print('HEAD:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import auth_check
import torch

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
assert os.environ['HF_TOKEN'], 'Missing HF_TOKEN'
assert os.environ['GITHUB_TOKEN'], 'Missing GITHUB_TOKEN'
assert torch.cuda.is_available(), 'CUDA required for canonical M3L-2'
assert torch.cuda.device_count() >= 2, f'M3L-2 requires two GPUs, found {torch.cuda.device_count()}'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
auth_check('archelabsxyz/native-clm-v0', repo_type='model', token=os.environ['HF_TOKEN'], write=True)
print('HF model repository write preflight: PASS')

In [ ]:
run([
    sys.executable, 'scripts/research/fetch_native_clm_v0_m1_checkpoint.py',
    '--repo-id', 'archelabsxyz/native-clm-v0',
    '--filename', 'final-model.pt',
    '--expected-sha256', '91cc66f744c97e50105acbb7cdc328a95cb87a32c49baf5b0d6e462d4d4c4c7f',
    '--output', CHECKPOINT,
])
print((CHECKPOINT_DIR / 'provenance.json').read_text())

In [ ]:
# Prepare one pinned matched B→C→D snapshot plus the explicitly registered pre-continual TinyStories-train bootstrap.
run([sys.executable, 'scripts/research/prepare_native_clm_v0_m3l2_data.py', '--output-dir', DATA])
manifest = json.loads((DATA / 'manifest.json').read_text())
print(json.dumps({
    'format': manifest['format'],
    'stream': manifest['stream'],
    'bootstrap': manifest['bootstrap'],
    'dataset_revisions': manifest['dataset_revisions'],
}, indent=2))

In [ ]:
# Refuse to claim a second untouched formal decision after canonical evidence has been tracked.
tracked_decision = 'artifacts/experiments/native-clm-v0-m3l2-online-address-state/decision.json'
tracked = subprocess.run(['git', 'ls-files', '--error-unmatch', tracked_decision], capture_output=True).returncode == 0
assert not tracked, 'Canonical M3L-2 decision is already tracked; later runs are reproduction only.'
protocol = json.loads(Path('research/validations/native-clm-v0-m3l2-online-address-state/protocol.json').read_text())
assert protocol['formal_seeds'] == [74211, 74212, 74213]
assert set(protocol['formal_seeds']).isdisjoint(protocol['consumed_seeds_forbidden'])
formal = run([
    sys.executable, 'scripts/research/run_native_clm_v0_m3l2.py',
    '--formal',
    '--checkpoint', CHECKPOINT,
    '--data-dir', DATA,
    '--output-dir', OUT,
    '--devices', '0,1',
], check=False)
assert formal.returncode in (0, 2), f'unexpected formal runner return code {formal.returncode}'
print('formal runner return code:', formal.returncode)

In [ ]:
decision = json.loads((OUT / 'decision.json').read_text())
print(json.dumps({
    'status': decision['status'],
    'scientific_decision': decision['scientific_decision'],
    'protocol_sha256': decision['protocol_sha256'],
    'data_manifest_sha256': decision['data_manifest_sha256'],
    'completed_seeds': decision['completed_seeds'],
}, indent=2))
for seed in decision['seed_results']:
    print('seed', seed['seed'], 'pass=', seed['pass'], 'control_A=', seed['control_A_regression'], 'treatment_A=', seed['treatment_A_regression'], 'advantage=', seed['A_retention_advantage'], 'forgetting=', seed['treatment_mean_forgetting'])
    for gate, passed in seed['gates'].items():
        print('  ', gate, passed)

In [ ]:
# HF-first publication: upload all six final checkpoints and decision before Git-publishing lightweight evidence.
run([
    sys.executable, 'scripts/research/publish_native_clm_v0_m3l2.py',
    '--branch', BRANCH,
    '--output-dir', OUT,
    '--checkpoint-provenance', CHECKPOINT_DIR / 'provenance.json',
    '--data-manifest', DATA / 'manifest.json',
    '--hf-repo', 'archelabsxyz/native-clm-v0',
    '--require-hf-upload',
])
print('Published M3L-2 status:', decision['status'])